### Import libraries

In [2]:
from pathlib import Path
import json
import pandas as pd

### Load the cleaned dataset

In [3]:
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

df = pd.read_csv(
    ROOT / "../data/processed/cleaned_paddy_qa.csv",
    keep_default_na=False,
    encoding="utf-8-sig",
)

### Select model columns

In [4]:
model_df = df[
    [
        "qa_id",
        "pair_id",
        "question",
        "language",
        "category_en",
        "issue_id",
        "issue_en",
    ]
].copy()

### Remove empty questions and labels

In [5]:
model_df = model_df.rename(
    columns={"category_en": "label"}
)

model_df = model_df[
    model_df["question"].str.strip().ne("")
    & model_df["label"].str.strip().ne("")
].copy()

### Create category labels

In [ ]:
labels = sorted(model_df["label"].unique())

label_to_id = {
    label: index
    for index, label in enumerate(labels)
}


### Convert categories into numbers and save

In [14]:

model_df["label_id"] = (
    model_df["label"].map(label_to_id).astype(int)
)

output_path = ROOT / "data" / "processed" / "model_ready_paddy_qa.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

model_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

label_path = Path(
    "/models/label_map.json"
)

label_path.parent.mkdir(parents=True, exist_ok=True)

label_path.write_text(
    json.dumps(label_to_id, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

323